In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,cross_val_score,RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,confusion_matrix
import joblib

In [5]:
df=pd.read_csv(r"E:\---\project\Model_main\loan_data_main.csv")

In [6]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
1,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
2,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
3,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
4,LP001013,Male,Yes,0,Not Graduate,No,2333,1516.0,95.0,360.0,1.0,Urban,Y


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381 entries, 0 to 380
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            381 non-null    object 
 1   Gender             376 non-null    object 
 2   Married            381 non-null    object 
 3   Dependents         373 non-null    object 
 4   Education          381 non-null    object 
 5   Self_Employed      360 non-null    object 
 6   ApplicantIncome    381 non-null    int64  
 7   CoapplicantIncome  381 non-null    float64
 8   LoanAmount         381 non-null    float64
 9   Loan_Amount_Term   370 non-null    float64
 10  Credit_History     351 non-null    float64
 11  Property_Area      381 non-null    object 
 12  Loan_Status        381 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 38.8+ KB


In [8]:
df.shape

(381, 13)

In [9]:
df.isnull().sum()

Loan_ID               0
Gender                5
Married               0
Dependents            8
Education             0
Self_Employed        21
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount            0
Loan_Amount_Term     11
Credit_History       30
Property_Area         0
Loan_Status           0
dtype: int64

In [10]:
df.columns

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status'],
      dtype='object')

In [11]:
df = df.drop(columns=['Gender','Loan_ID', 'Dependents','Loan_Amount_Term'])

# df = df.drop(columns=['Loan_ID'])

df.head()

,Married,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Credit_History,Property_Area,Loan_Status
0,Yes,Graduate,No,4583,1508.0,128.0,1.0,Rural,N
1,Yes,Graduate,Yes,3000,0.0,66.0,1.0,Urban,Y
2,Yes,Not Graduate,No,2583,2358.0,120.0,1.0,Urban,Y
3,No,Graduate,No,6000,0.0,141.0,1.0,Urban,Y
4,Yes,Not Graduate,No,2333,1516.0,95.0,1.0,Urban,Y


In [12]:
df.isnull().sum()

Married               0
Education             0
Self_Employed        21
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount            0
Credit_History       30
Property_Area         0
Loan_Status           0
dtype: int64

In [13]:
cols=['Self_Employed','Credit_History']

for col in cols:
    print(df[col].unique())

['No' 'Yes' nan]
[ 1. nan  0.]


In [14]:
df['Self_Employed'].mode()

0    No
Name: Self_Employed, dtype: object

In [15]:
df['Credit_History'].mode()

0    1.0
Name: Credit_History, dtype: float64

In [16]:
df['Self_Employed'].fillna(df['Self_Employed'].mode()[0], inplace=True)
df['Credit_History'].fillna(df['Credit_History'].mode()[0], inplace=True)

In [17]:
df.isnull().sum()

Married              0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381 entries, 0 to 380
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Married            381 non-null    object 
 1   Education          381 non-null    object 
 2   Self_Employed      381 non-null    object 
 3   ApplicantIncome    381 non-null    int64  
 4   CoapplicantIncome  381 non-null    float64
 5   LoanAmount         381 non-null    float64
 6   Credit_History     381 non-null    float64
 7   Property_Area      381 non-null    object 
 8   Loan_Status        381 non-null    object 
dtypes: float64(3), int64(1), object(5)
memory usage: 26.9+ KB


In [19]:
df.columns

Index(['Married', 'Education', 'Self_Employed', 'ApplicantIncome',
       'CoapplicantIncome', 'LoanAmount', 'Credit_History', 'Property_Area',
       'Loan_Status'],
      dtype='object')

In [20]:
encoding = {
    'Married':{'Yes':1,'No':0},
    'Education':{'Graduate':1, 'Not Graduate':0},
    'Self_Employed':{'No':0 ,'Yes':1},
    'Property_Area':{'Rural':0, 'Urban':1 ,'Semiurban':2},
    'Loan_Status':{'N':0 ,'Y':1}
}
df.replace(encoding, inplace=True)

C:\Users\Yash\AppData\Local\Temp\ipykernel_14296\704786972.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(encoding, inplace=True)


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381 entries, 0 to 380
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Married            381 non-null    int64  
 1   Education          381 non-null    int64  
 2   Self_Employed      381 non-null    int64  
 3   ApplicantIncome    381 non-null    int64  
 4   CoapplicantIncome  381 non-null    float64
 5   LoanAmount         381 non-null    float64
 6   Credit_History     381 non-null    float64
 7   Property_Area      381 non-null    int64  
 8   Loan_Status        381 non-null    int64  
dtypes: float64(3), int64(6)
memory usage: 26.9 KB


In [22]:
df.tail()

,Married,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Credit_History,Property_Area,Loan_Status
376,1,1,0,5703,0.0,128.0,1.0,1,1
377,1,1,0,3232,1950.0,108.0,1.0,0,1
378,0,1,0,2900,0.0,71.0,1.0,0,1
379,1,1,0,4106,0.0,40.0,1.0,0,1
380,0,1,1,4583,0.0,133.0,0.0,2,0


In [23]:
df.columns

Index(['Married', 'Education', 'Self_Employed', 'ApplicantIncome',
       'CoapplicantIncome', 'LoanAmount', 'Credit_History', 'Property_Area',
       'Loan_Status'],
      dtype='object')

In [24]:
x = df.drop('Loan_Status',axis=1)
y = df['Loan_Status']

In [25]:
num_cols=['ApplicantIncome','CoapplicantIncome','LoanAmount','Credit_History']
scaler=StandardScaler()
x[num_cols]=scaler.fit_transform(x[num_cols])

In [26]:
x.head()

,Married,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Credit_History,Property_Area
0,1,1,0,0.707469,0.098695,0.812575,0.419435,0
1,1,1,1,-0.408932,-0.546371,-1.376596,0.419435,1
2,1,0,0,-0.703019,0.462294,0.530102,0.419435,1
3,0,1,0,1.706799,-0.546371,1.271595,0.419435,1
4,1,0,0,-0.879330,0.102118,-0.352629,0.419435,1


In [27]:
def evaluate_model(model):
    x_train, x_test, y_train, y_test =train_test_split(x, y, test_size=0.2, random_state=42)
    model.fit(x_train,y_train)
    y_pred = model.predict(x_test)
    accuracy=accuracy_score(y_test,y_pred)
    cross_val= cross_val_score(model, x, y, cv=5)
    avg_cross_val=np.mean(cross_val)
    print(f"{model.__class__.__name__} - Accuracy:{accuracy: .2f}, Cross_val_score :{avg_cross_val: .2f} ")
    return avg_cross_val

In [28]:
models={ 
    LogisticRegression(),
    svm.SVC(),
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    GradientBoostingClassifier(),
}

In [29]:
model_score = {model.__class__.__name__: evaluate_model(model) for model in models}

GradientBoostingClassifier - Accuracy: 0.81, Cross_val_score : 0.82 
DecisionTreeClassifier - Accuracy: 0.69, Cross_val_score : 0.79 
RandomForestClassifier - Accuracy: 0.83, Cross_val_score : 0.82 
SVC - Accuracy: 0.82, Cross_val_score : 0.85 
LogisticRegression - Accuracy: 0.82, Cross_val_score : 0.84 


In [30]:
def tune_model(model, param_grid):
    tuner=RandomizedSearchCV(model, param_grid, cv=5, n_iter=20, verbose=True, random_state=42 )
    tuner.fit(x,y)
    print(f"Best score for {model.__class__.__name__}: {tuner.best_score_:.2f}")
    print(f"Best parameter for {model.__class__.__name__}: {tuner.best_params_}")
    return tuner.best_estimator_

In [31]:
log_reg_grid = {
    'C': np.logspace(-4, 4, 20),
    'solver': ['liblinear']
}

svc_grid = {
    'C': [0.25, 0.50, 0.75, 1],
    'kernel': ['linear']         
}

rf_grid = {
    'n_estimators': np.arange(10, 1000, 10),
    'max_features': ['log2', 'sqrt'],  
    'max_depth': [None, 3, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10, 20, 50, 100],
    'min_samples_leaf': [1, 2, 5, 10]
}


In [32]:
best_log_reg=tune_model(LogisticRegression(),log_reg_grid)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best score for LogisticRegression: 0.85
Best parameter for LogisticRegression: {'solver': 'liblinear', 'C': np.float64(0.012742749857031334)}


In [33]:
best_svc=tune_model(svm.SVC(),svc_grid)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best score for SVC: 0.85
Best parameter for SVC: {'kernel': 'linear', 'C': 0.25}


c:\Users\Yash\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=20. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [34]:
best_rf=tune_model(RandomForestClassifier(),rf_grid)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best score for RandomForestClassifier: 0.85
Best parameter for RandomForestClassifier: {'n_estimators': np.int64(70), 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 3}


In [35]:
final_model = best_rf

In [36]:
joblib.dump(final_model, "loan_status_predictor.pkl")

['loan_status_predictor.pkl']

In [37]:
# prediction system
sample_data = pd.DataFrame({
    'Married': [0],
    'Education': [1],
    'Self_Employed': [1],
    'ApplicantIncome': [4583],
    'CoapplicantIncome': [0.0],
    'LoanAmount': [133.0],
    'Credit_History': [0.0],
    'Property_Area': [2]
})
sample_data[num_cols]=scaler.transform(sample_data[num_cols])
loaded_model= joblib.load('loan_status_predictor.pkl')
prediction = loaded_model.predict(sample_data)

result = "Loan Approved" if prediction[0]==1 else "Loan Not Approved"
print(f"\nPrediction Result: {result}")


Prediction Result: Loan Not Approved


In [38]:
joblib.dump(scaler,'vector.pkl')

['vector.pkl']

In [39]:
df.columns

Index(['Married', 'Education', 'Self_Employed', 'ApplicantIncome',
       'CoapplicantIncome', 'LoanAmount', 'Credit_History', 'Property_Area',
       'Loan_Status'],
      dtype='object')